In [0]:
%sql
use catalog marketing_analytics_capstone;
USE schema bronze;

In [0]:
%sql
SELECT
    source_month,
    COUNT(*)              AS total_records,
    COUNT(DISTINCT batch_id) AS total_batches,
    MIN(ingestion_timestamp) AS first_loaded,
    MAX(ingestion_timestamp) AS last_loaded
FROM marketing_analytics_capstone.bronze.bronze_campaigns
GROUP BY source_month
ORDER BY source_month;

### Understanding the Schema

In [0]:
%sql
DESCRIBE TABLE marketing_analytics_capstone.bronze.bronze_campaigns;

In [0]:
spark.read.table("marketing_analytics_capstone.bronze.bronze_campaigns").printSchema()

### Basic profiling (shape of data)

In [0]:
%sql
SELECT COUNT(*) FROM marketing_analytics_capstone.bronze.bronze_campaigns;

In [0]:
%sql
SELECT * 
FROM marketing_analytics_capstone.bronze.bronze_campaigns
LIMIT 10;

### Null analysis

In [0]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(Campaign_Type) AS non_null_campaign_name,
    COUNT(Date) AS non_null_date,
    COUNT(clicks) AS non_null_clicks,
    COUNT(impressions) AS non_null_impressions
FROM marketing_analytics_capstone.bronze.bronze_campaigns;

In [0]:
%sql
SELECT
    SUM(CASE WHEN Campaign_Type IS NULL THEN 1 ELSE 0 END) AS null_campaign_name,
    SUM(CASE WHEN Date IS NULL THEN 1 ELSE 0 END) AS null_date,
    SUM(CASE WHEN clicks IS NULL THEN 1 ELSE 0 END) AS null_clicks,
    SUM(CASE WHEN impressions IS NULL THEN 1 ELSE 0 END) AS null_impressions
FROM marketing_analytics_capstone.bronze.bronze_campaigns;

### Duplicate detection

In [0]:
%sql
SELECT Campaign_Type, Date, COUNT(*) as cnt
FROM marketing_analytics_capstone.bronze.bronze_campaigns
GROUP BY Campaign_Type, Date
HAVING COUNT(*) > 1;

In [0]:
%sql
SELECT COUNT(*) AS total,
       COUNT(DISTINCT Campaign_Type, Date) AS unique_records
FROM marketing_analytics_capstone.bronze.bronze_campaigns;

### Distinct values (categorical cleaning)

In [0]:
%sql
SELECT DISTINCT Channel_Used
FROM marketing_analytics_capstone.bronze.bronze_campaigns
ORDER BY Channel_Used;

In [0]:
%sql
SELECT Channel_Used, COUNT(*) 
FROM marketing_analytics_capstone.bronze.bronze_campaigns
GROUP BY Channel_Used
ORDER BY COUNT(*) DESC;

### Date validation

In [0]:
%sql
SELECT Date, COUNT(*) 
FROM marketing_analytics_capstone.bronze.bronze_campaigns
GROUP BY Date
ORDER BY Date;

In [0]:
%sql
SELECT *
FROM marketing_analytics_capstone.bronze.bronze_campaigns
WHERE to_date(Date, 'yyyy-MM-dd') IS NULL;

### Numerical anomalies

In [0]:
%sql
SELECT
    MIN(clicks) AS min_clicks,
    MAX(clicks) AS max_clicks,
    MIN(impressions) AS min_impressions,
    MAX(impressions) AS max_impressions
FROM marketing_analytics_capstone.bronze.bronze_campaigns;

In [0]:
%sql
SELECT *
FROM marketing_analytics_capstone.bronze.bronze_campaigns
WHERE clicks < 0 OR impressions < 0;

In [0]:
%sql
SELECT *
FROM marketing_analytics_capstone.bronze.bronze_campaigns
WHERE clicks > impressions;

### Distribution check

In [0]:
%sql
SELECT impressions, COUNT(*) 
FROM marketing_analytics_capstone.bronze.bronze_campaigns
GROUP BY impressions
ORDER BY impressions DESC;

In [0]:
%sql
SELECT percentile_approx(impressions, 0.5) AS median,
       percentile_approx(impressions, 0.9) AS p90,
       percentile_approx(impressions, 0.99) AS p99
FROM marketing_analytics_capstone.bronze.bronze_campaigns;

### Batch consistency check

In [0]:
%sql
SELECT batch_id, COUNT(*) 
FROM marketing_analytics_capstone.bronze.bronze_campaigns
GROUP BY batch_id;

### Source-level validation

In [0]:
%sql
SELECT source_file_name, COUNT(*) 
FROM marketing_analytics_capstone.bronze.bronze_campaigns
GROUP BY source_file_name;

### Check rescued data

In [0]:
%sql
SELECT *
FROM marketing_analytics_capstone.bronze.bronze_campaigns
WHERE _rescued_data IS NOT NULL;

### Validate Acquisition Cost format

In [0]:
%sql
SELECT Acquisition_Cost
FROM marketing_analytics_capstone.bronze.bronze_campaigns
WHERE Acquisition_Cost NOT LIKE '$%';

### Conversion rate sanity check

In [0]:
%sql
SELECT *
FROM marketing_analytics_capstone.bronze.bronze_campaigns
WHERE Conversion_Rate < 0 OR Conversion_Rate > 1;

### ROI Sanity Check

In [0]:
%sql
SELECT *
FROM marketing_analytics_capstone.bronze.bronze_campaigns
WHERE ROI < 0;

### Logical consistency: clicks vs impressions

In [0]:
%sql
SELECT *
FROM marketing_analytics_capstone.bronze.bronze_campaigns
WHERE Clicks > Impressions;

### Duration parsing check

In [0]:
%sql
SELECT DISTINCT Duration
FROM marketing_analytics_capstone.bronze.bronze_campaigns;

### Channel vs Campaign Type consistency

In [0]:
%sql
SELECT Campaign_Type, Channel_Used, COUNT(*)
FROM marketing_analytics_capstone.bronze.bronze_campaigns
GROUP BY Campaign_Type, Channel_Used
ORDER BY COUNT(*) DESC;

### Categorical cleaning check

In [0]:
%sql
SELECT DISTINCT Channel_Used FROM marketing_analytics_capstone.bronze.bronze_campaigns;

In [0]:
%sql
SELECT DISTINCT Campaign_Type FROM marketing_analytics_capstone.bronze.bronze_campaigns;

In [0]:
%sql
SELECT DISTINCT Language FROM marketing_analytics_capstone.bronze.bronze_campaigns;

In [0]:
%sql
SELECT DISTINCT Location FROM marketing_analytics_capstone.bronze.bronze_campaigns;

### Outlier detection (cost vs impressions)

In [0]:
%sql
SELECT *
FROM marketing_analytics_capstone.bronze.bronze_campaigns
WHERE Impressions > 9000 AND Acquisition_Cost < '$5000';

### Batch duplication check (important for streaming)

In [0]:
%sql
SELECT Campaign_ID, COUNT(DISTINCT batch_id) as batch_count
FROM marketing_analytics_capstone.bronze.bronze_campaigns
GROUP BY Campaign_ID
HAVING COUNT(DISTINCT batch_id) > 1;